## Построение пайплайнов для загрузки параметров

### Загрузка всех датасетов

В начале загрузим все обработанные версии датасетов.

In [1]:
!pip install scikit-optimize
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.0 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error, make_scorer
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df_adult = pd.read_csv('/content/adult_cleaned_new.csv')
df_bank = pd.read_csv('/content/bank_marketing_cleaned_new.csv')
df_cal = pd.read_csv('/content/california_housing_cleaned_new.csv')
df_sc = pd.read_csv('/content/superconductivity_cleaned_new.csv')
df_sp = pd.read_csv('/content/spambase_cleaned_new.csv')

In [ ]:
datasets = {
    'Adult': df_adult,
    'Bank': df_bank,
    'California': df_cal,
    'Superconductivity': df_sc,
    'Spam': df_sp
}

In [ ]:
for name, data in datasets.items():
  print(f'{name}:', end = '\n')
  print(len(data))

Adult:
48842
Bank:
45211
California:
20640
Superconductivity:
21263
Spam:
4601


In [ ]:
for name, data in datasets.items():
  print(f'{name}:', end = '\n')
  print(data['target'].nunique())

Adult:
2
Bank:
2
California:
3842
Superconductivity:
3007
Spam:
2


### Создаем конфиги

**Конфиги датасетов**

В конфиге задаем:

1. Какой тип задачи решается в датасете
2. Колонку с таргетом
3. Колонки с категориальными и числовыми фичами
4. Флаг того, большой датасет или маленький (в зависимости от этого определяется число фолдов при оценке качества моделей).

In [ ]:
DATASET_CONFIGS = {
    'Adult': {
        'dataframe': df_adult,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': ['workclass', 'education', 'marital-status', 'occupation',
                             'relationship', 'race', 'sex', 'native-country'],
        'numeric_cols': ['age', 'fnlwgt', 'education-num', 'capital-gain',
                        'capital-loss', 'hours-per-week'],
        'size': 'large'
    },
    'Bank': {
        'dataframe': df_bank,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': ['job', 'marital', 'education', 'contact',
                            'day_of_week', 'month', 'poutcome'],
        'numeric_cols': ['age', 'balance', 'campaign', 'pdays', 'previous',
                        'default', 'housing', 'loan', 'was_contacted'],
        'size': 'large'
    },
    'California': {
        'dataframe': df_cal,
        'task': 'regression',
        'target_col': 'target',
        'categorical_cols': ['ocean_proximity'],
        'numeric_cols': ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                        'total_bedrooms', 'population', 'households', 'median_income',
                        'rooms_per_household', 'bedrooms_per_room', 'population_per_household'],
        'size': 'large'
    },
    'Superconductivity': {
        'dataframe': df_sc,
        'task': 'regression',
        'target_col': 'target',
        'categorical_cols': [],
        'numeric_cols': None,  # Все колонки кроме target
        'size': 'large'
    },
    'Spam': {
        'dataframe': df_sp,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': [],
        'numeric_cols': None,  # Все колонки кроме target
        'size': 'small'
    }
}

**Конфиги моделей**

В конфиге задаем:

1. Какой тип модели используем
2. Словарь с перебираемыми параметрами
3. Список поддерживаемых задач

In [ ]:
#Конфиги моделей с перебираемыми параметрами
MODEL_CONFIGS = {
    'LogisticRegression': {
        'class': LogisticRegression,
        'params': {
            'C': Real(0.01, 100, prior='log-uniform'),
            'penalty': Categorical(['l1', 'l2']),
            #'penalty': Categorical(['l1', 'l2']),
            #'solver': Categorical(['liblinear', 'saga']),
            'solver': Categorical(['liblinear']),
            'max_iter': Integer(100, 1000),
            'tol': [1e-3]
        },
        'supports': ['classification']
    },
    'Ridge': {
        'class': Ridge,
        'params': {
            'alpha': Real(0.001, 100, prior='log-uniform'),
            'solver': Categorical(['auto', 'svd', 'cholesky', 'lsqr', 'sag'])
        },
        'supports': ['regression']
    },
    'Lasso': {
        'class': Lasso,
        'params': {
            'alpha': Real(0.0001, 10, prior='log-uniform'),
            'max_iter': Integer(1000, 5000),
            'selection': Categorical(['cyclic', 'random'])
        },
        'supports': ['regression']
    },

    'ElasticNet': {
        'class': ElasticNet,
        'params': {
            'alpha': Real(0.001, 10, prior='log-uniform'),
            'l1_ratio': Real(0.1, 0.9),  # 0 = Ridge, 1 = Lasso
            'max_iter': Integer(1000, 5000)
        },
        'supports': ['regression']
    },
    'RandomForestClassifier': {
        'class': RandomForestClassifier,
        'params': {
            'n_estimators': Integer(50, 300),
            'max_depth': Integer(3, 20),
            'min_samples_split': Integer(2, 20),
            'min_samples_leaf': Integer(1, 10),
            'max_features': Categorical(['sqrt', 'log2'])
            #'max_features': Categorical(['sqrt', 'log2', None])
        },
        'supports': ['classification']
    },
    'RandomForestRegressor': {
        'class': RandomForestRegressor,
        'params': {
            #'n_estimators': Integer(50, 500),
            'n_estimators': Integer(50, 300),
            'max_depth': Integer(3, 20),
            'min_samples_split': Integer(2, 20),
            'min_samples_leaf': Integer(1, 10),
            'max_features': Categorical(['sqrt', 'log2'])
            #'max_features': Categorical(['sqrt', 'log2', None])
        },
        'supports': ['regression']
    },
    'LightGBMClassifier': {
        'class': lgb.LGBMClassifier,
        'params': {
            'n_estimators': Integer(50, 500),
            'max_depth': Integer(3, 15),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'num_leaves': Integer(10, 100),
            'min_child_samples': Integer(5, 50),
            'subsample': Real(0.6, 1.0),
            'colsample_bytree': Real(0.6, 1.0),
            'verbose': [-1]
        },
        'supports': ['classification']
    },
    'LightGBMRegressor': {
        'class': lgb.LGBMRegressor,
        'params': {
            'n_estimators': Integer(50, 500),
            'max_depth': Integer(3, 15),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'num_leaves': Integer(10, 100),
            'min_child_samples': Integer(5, 50),
            'subsample': Real(0.6, 1.0),
            'colsample_bytree': Real(0.6, 1.0),
            'verbose': [-1]
        },
        'supports': ['regression']
    },
    'CatBoostClassifier': {
        'class': CatBoostClassifier,
        'params': {
            'iterations': Integer(50, 500),
            'depth': Integer(3, 10),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'l2_leaf_reg': Real(1, 10),
            'border_count': Integer(32, 255),
            'verbose': [0]
        },
        'supports': ['classification']
    },
    'CatBoostRegressor': {
        'class': CatBoostRegressor,
        'params': {
            'iterations': Integer(50, 500),
            'depth': Integer(3, 10),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'l2_leaf_reg': Real(1, 10),
            'border_count': Integer(32, 255),
            'verbose': [0]
        },
        'supports': ['regression']
    }
}

### Пишем препроцессинг

In [ ]:
def get_preprocessor(dataset_name, model_type):
    """
    Создает препроцессор в зависимости от датасета и типа модели.

    Args:
        dataset_name: Название датасета
        model_type: Тип модели ('linear', 'tree', 'boosting')
    Returns:
        ColumnTransformer или None
    """
    config = DATASET_CONFIGS[dataset_name]
    categorical_cols = config['categorical_cols']
    numeric_cols = config['numeric_cols']

    # Если все признаки числовые
    if not categorical_cols and numeric_cols is None:
        if model_type == 'linear':
            return RobustScaler()  # Устойчив к выбросам
        else:
            return None  # Деревьям скейлинг не нужен

    # Если есть категориальные признаки
    if model_type == 'linear':
        # Для линейных моделей: OneHotEncoder + RobustScaler
        return ColumnTransformer([
            ('num', RobustScaler(), numeric_cols if numeric_cols else
             [col for col in config['dataframe'].columns
              if col != config['target_col'] and col not in categorical_cols]),
            ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
             categorical_cols)
        ])

    elif model_type == 'tree':
        return ColumnTransformer([
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
             categorical_cols)
        ], remainder='passthrough')

    elif model_type == 'boosting':
        # Для LightGBM/CatBoost: ничего не делаем, они сами обработают категории
        return None


def prepare_data(dataset_name):
    """
    Подготавливает X и y для датасета.
    """
    config = DATASET_CONFIGS[dataset_name]
    df = config['dataframe']
    target_col = config['target_col']

    X = df.drop(target_col, axis=1)
    y = df[target_col]

    # Для датасетов без явных числовых колонок берем все колонки
    if config['numeric_cols'] is None:
        config['numeric_cols'] = list(X.select_dtypes(include=[np.number]).columns)

    return X, y

Выберем дополнительно тип кросс-валидации в зависимости от датасета.

Также сделаем маппинг модели и типа задачи, которую она решает, с классом в sklearn.

In [ ]:
def get_cv_splitter(dataset_name, task):
    """
    Выбирает тип кросс-валидации в зависимости от размера датасета и задачи.
    """
    config = DATASET_CONFIGS[dataset_name]

    if config['size'] == 'large':
        n_splits = 3
    else:
        n_splits = 3

    if task == 'classification':
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    else:
        return KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
def get_model_for_task(model_name, task):
    """
    Подбирает правильную модель в зависимости от задачи.
    """
    mapping = {
        # Для классификации
        ('LogisticRegression', 'classification'): 'LogisticRegression',
        ('RandomForest', 'classification'): 'RandomForestClassifier',
        ('LightGBM', 'classification'): 'LightGBMClassifier',
        ('CatBoost', 'classification'): 'CatBoostClassifier',

        # Для регрессии
        #('LogisticRegression', 'regression'): 'Ridge',  # По умолчанию Ridge
        ('Ridge', 'regression'): 'Ridge',
        ('Lasso', 'regression'): 'Lasso',
        ('ElasticNet', 'regression'): 'ElasticNet',
        ('RandomForest', 'regression'): 'RandomForestRegressor',
        ('LightGBM', 'regression'): 'LightGBMRegressor',
        ('CatBoost', 'regression'): 'CatBoostRegressor'
    }

    return mapping.get((model_name, task))


def get_model_type(model_name):
    """
    Определяет тип модели для выбора препроцессора.
    """
    if model_name in ['LogisticRegression', 'Ridge', 'Lasso', 'ElasticNet']:
        return 'linear'
    elif model_name == 'RandomForest':
        return 'tree'
    elif model_name in ['LightGBM', 'CatBoost']:
        return 'boosting'
    else:
        # На всякий случай: если передано полное имя модели
        if 'Logistic' in model_name:
            return 'linear'
        elif 'Ridge' in model_name or 'Lasso' in model_name or 'ElasticNet' in model_name:
            return 'linear'
        elif 'RandomForest' in model_name:
            return 'tree'
        elif 'LightGBM' in model_name or 'CatBoost' in model_name:
            return 'boosting'
        else:
            raise ValueError(f"Неизвестный тип модели: {model_name}")

In [ ]:
## Функция для логирования
import csv
import os

def log_best_to_csv(log_file, iteration, best_score, best_params, task, metric_name):
    """Дописывает строку в CSV-файл логов."""
    file_exists = os.path.isfile(log_file)
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['iteration', 'best_score', 'best_params', 'task', 'metric'])
        writer.writerow([iteration, best_score, best_params, task, metric_name])

### Основная функция для оптимизации

Напишем функцию, которая принимает на вход:

1. Датасет
2. Модель
3. Число Итераций

И ищет наилучшие параметры модели по байесовскому подходу, используя гауссовский случайный процесс.

In [ ]:
def run_bayesian_optimization(dataset_name, model_name, n_iter=30,
                              log_interval=5, log_file=None):
    """
    Запускает байесовскую оптимизацию гиперпараметров.
    Для CatBoost используется отдельная реализация.
    """
    dataset_config = DATASET_CONFIGS[dataset_name]
    task = dataset_config['task']

    full_model_name = get_model_for_task(model_name, task)
    if full_model_name not in MODEL_CONFIGS:
        raise ValueError(f"Модель {model_name} не поддерживает задачу {task}")

    model_config = MODEL_CONFIGS[full_model_name]

    print(f"\n{'='*70}")
    print(f"Байесовская оптимизация: {model_name} на {dataset_name}")
    print(f"Задача: {task.upper()}")
    print(f"{'='*70}")

    # Подготовка данных
    X, y = prepare_data(dataset_name)
    print(f"Размер данных: X={X.shape}, y={y.shape}")

    # Если CatBoost – передаём в специальную функцию
    '''if model_name == 'CatBoost':
        cv = get_cv_splitter(dataset_name, task)
        scoring = 'roc_auc' if task == 'classification' else 'neg_mean_squared_error'
        best_score, best_params, opt_result = run_catboost_optimization(
            dataset_name, model_name, task, X, y, cv, scoring, n_iter
        )
        results = {
            'dataset': dataset_name,
            'model': model_name,
            'task': task,
            'best_score': best_score,
            'best_params': best_params,
            'cv_results': opt_result
        }
        print(f"\n{'='*70}")
        print(f"РЕЗУЛЬТАТЫ:")
        print(f"Лучшее значение метрики: {best_score:.4f}")
        print(f"Лучшие параметры:")
        for param, value in best_params.items():
            print(f"  {param}: {value}")
        print(f"{'='*70}")
        return results'''

    # Для всех остальных моделей – стандартный Pipeline + BayesSearchCV
    model_type = get_model_type(model_name)
    preprocessor = get_preprocessor(dataset_name, model_type)

    if preprocessor is not None:
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model_config['class']())
        ])
    else:
        pipeline = Pipeline([
            ('model', model_config['class']())
        ])

    params = {f'model__{key}': value for key, value in model_config['params'].items()}

    scoring = 'roc_auc' if task == 'classification' else 'neg_mean_squared_error'
    metric_name = 'ROC-AUC' if task == 'classification' else 'MSE'

    cv = get_cv_splitter(dataset_name, task)
    print(f"Кросс-валидация: {cv.get_n_splits()}-fold")
    print(f"Метрика: {metric_name}")

    # Базовая проверка
    print("\nПроверка базовой кросс-валидации...")
    try:
        base_scores = cross_val_score(
            pipeline, X, y,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            error_score='raise'
        )
        print(f"Базовая {metric_name}: {base_scores.mean():.4f} (+/- {base_scores.std() * 2:.4f})")
    except Exception as e:
        print(f"Ошибка при базовой проверке: {e}")

    # Запуск BayesSearchCV
    print(f"\nЗапуск оптимизации ({n_iter} итераций)...")
    bayes_search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=params,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        random_state=42,
        n_jobs=1,
        n_points=1,
        verbose=10,
        pre_dispatch=1
    )

    bayes_search.fit(X, y)

    if log_file and model_name != 'CatBoost':
        cv_results = bayes_search.cv_results_
        # Порядок итераций соответствует порядку в cv_results_
        mean_scores = cv_results['mean_test_score']  # средние по фолдам
        params_list = cv_results['params']
        # Определяем лучший скор и параметры нарастающим итогом
        best_score_so_far = -np.inf
        best_params_so_far = None
        for i, (score, par) in enumerate(zip(mean_scores, params_list)):
            if score > best_score_so_far:
                best_score_so_far = score
                best_params_so_far = par
            iteration = i + 1  # итерации с 1
            if iteration % log_interval == 0:
                metric_name = 'ROC-AUC' if task=='classification' else 'MSE'
                log_best_to_csv(log_file, iteration, best_score_so_far,
                                best_params_so_far, task, metric_name)

    results = {
        'dataset': dataset_name,
        'model': model_name,
        'task': task,
        'best_score': bayes_search.best_score_,
        'best_params': bayes_search.best_params_,
        'cv_results': bayes_search.cv_results_
    }

    print(f"\n{'='*70}")
    print(f"РЕЗУЛЬТАТЫ:")
    print(f"Лучшее значение метрики: {bayes_search.best_score_:.4f}")
    print(f"Лучшие параметры:")
    for param, value in bayes_search.best_params_.items():
        print(f"  {param}: {value}")
    print(f"{'='*70}")

    return results

Возможные модели:

1. LogisticRegression
2. Ridge
3. Lasso
4. ElasticNet
5. RandomForest
6. LightGBM
7. CatBoost

Возможные датасеты:

1. Adult
2. Bank
3. California
4. Superconductivity
5. Spam

Добавим логирование на Гугл Диск

In [3]:
from google.colab import drive
import os

# Монтируем Google Диск
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Задаём папку для логов на Диске (можно изменить название)
base_log_dir = '/content/drive/MyDrive/ЦУ Курсы/Метопты/Bayes_logs'
os.makedirs(base_log_dir, exist_ok=True)  # создаём, если не существует

Запустим функцию

In [ ]:
log_interval = 5
log_file = os.path.join(base_log_dir, 'bayes_RandomForest_Superconductivity_logs.csv')

test_results = run_bayesian_optimization('Superconductivity', 'RandomForest', n_iter=50,
                                        log_interval = log_interval, log_file = log_file)


Байесовская оптимизация: RandomForest на Spam
Задача: CLASSIFICATION
Размер данных: X=(4601, 57), y=(4601,)
Кросс-валидация: 3-fold
Метрика: ROC-AUC

Проверка базовой кросс-валидации...
Базовая ROC-AUC: 0.9839 (+/- 0.0053)

Запуск оптимизации (50 итераций)...
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV 1/3; 1/1] START model__max_depth=10, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218
[CV 1/3; 1/1] END model__max_depth=10, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218;, score=0.980 total time=   1.1s
[CV 2/3; 1/1] START model__max_depth=10, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218
[CV 2/3; 1/1] END model__max_depth=10, model__max_features=log2, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218;, score=0.977 total time=   1.1s
[CV 3/3; 1/1] START model__max_dep

### Поиск по одной модели для всех датасетов

In [ ]:
import os

def optimize_model_on_all_datasets(model_name, n_iter=50, log_interval=5, base_log_dir='logs'):
    """
    Запускает run_bayesian_optimization для выбранной модели на всех датасетах,
    где задача (классификация/регрессия) поддерживается моделью.

    Параметры
    ---------
    model_name : str
        Имя модели ('LogisticRegression', 'RandomForest', 'LightGBM', 'CatBoost',
                    'Ridge', 'Lasso', 'ElasticNet').
    n_iter : int
        Число итераций оптимизации.
    log_interval : int
        Интервал сохранения логов (каждые N итераций).
    base_log_dir : str
        Папка, в которую будут сохраняться логи.
    """
    # Создаём папку для логов, если её нет
    os.makedirs(base_log_dir, exist_ok=True)

    # Список всех датасетов
    all_datasets = list(DATASET_CONFIGS.keys())

    for dataset_name in all_datasets:
        # Определяем задачу датасета
        task = DATASET_CONFIGS[dataset_name]['task']

        # Проверяем, поддерживает ли модель эту задачу
        if get_model_for_task(model_name, task) is None:
            print(f"Пропускаем {dataset_name} – модель {model_name} не поддерживает задачу {task}")
            continue

        # Формируем имя файла логов
        log_filename = f"bayes_{model_name}_{dataset_name}_logs.csv"
        log_filepath = os.path.join(base_log_dir, log_filename)

        print(f"\n====== Запуск {model_name} на {dataset_name} ======")

        # Запускаем оптимизацию (внутри уже есть запись логов)
        run_bayesian_optimization(
            dataset_name=dataset_name,
            model_name=model_name,
            n_iter=n_iter,
            log_interval=log_interval,
            log_file=log_filepath
        )

In [ ]:
optimize_model_on_all_datasets('RandomForest', n_iter=50, log_interval=5, base_log_dir='/content/drive/MyDrive/ЦУ Курсы/Метопты/Bayes_logs')


====== Запуск RandomForest на Adult ======

Байесовская оптимизация: RandomForest на Adult
Задача: CLASSIFICATION
Размер данных: X=(48842, 14), y=(48842,)
Кросс-валидация: 3-fold
Метрика: ROC-AUC

Проверка базовой кросс-валидации...
Базовая ROC-AUC: 0.9064 (+/- 0.0020)

Запуск оптимизации (50 итераций)...
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV 1/3; 1/1] START model__max_depth=10, model__max_features=None, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218
[CV 1/3; 1/1] END model__max_depth=10, model__max_features=None, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218;, score=0.913 total time=  24.2s
[CV 2/3; 1/1] START model__max_depth=10, model__max_features=None, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218
[CV 2/3; 1/1] END model__max_depth=10, model__max_features=None, model__min_samples_leaf=9, model__min_samples_split=8, model__n_estimators=218;, score=0.915 total 

KeyboardInterrupt: 

### Архив